# FraudShield AI: DistilBERT hyperparameter sweep (Colab)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 20–25 minutes.

What this does:
1. Gets the repo (clone, or upload a zip if the repo is private).
2. Installs only what's missing. Colab's own PyTorch (with CUDA) is kept.
3. Rebuilds the message split and **checks it matches the committed split exactly** (stops if not).
4. Runs the sweep: learning rate {2e-5, 3e-5, 5e-5} × max length {64, 128}, choosing the best by **validation** macro-F1. The test set is not touched.
5. Packs the best model and its metrics into `distilbert_sweep_results.zip` for download (and optionally copies it to Drive).

Then follow **"After the Colab sweep"** in the README.

In [ ]:
!nvidia-smi -L
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"


### 1. Get the code
**Public repo:** run the next cell as is.
**Private repo:** either add a GitHub token as a Colab secret named `GH_TOKEN` (🔑 icon on the left) and set `USE_TOKEN = True`, or upload a zip of the repo and set `UPLOADED_ZIP = True`.

In [ ]:
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of asking for a password
USE_TOKEN = False
UPLOADED_ZIP = False
REPO = "github.com/Moeez-abid/FraudShieldAI.git"

os.chdir("/content")
if UPLOADED_ZIP:
    from google.colab import files
    up = files.upload()                      # choose the repo zip
    zip_name = next(iter(up))
    subprocess.run(["unzip", "-q", "-o", zip_name], check=True)
    root = next(d for d in os.listdir() if os.path.isfile(os.path.join(d, "pyproject.toml")))
elif not os.path.isdir("FraudShieldAI"):
    url = "https://" + REPO
    if USE_TOKEN:
        from google.colab import userdata
        url = f"https://{userdata.get('GH_TOKEN')}@{REPO}"
    subprocess.run(["git", "clone", "-q", url, "FraudShieldAI"], check=True)
    root = "FraudShieldAI"
else:
    root = "FraudShieldAI"
os.chdir(f"/content/{root}")
print(os.getcwd()); print(open("README.md").readline())


### 2. Install (keeps Colab's PyTorch)

In [ ]:
!pip -q install transformers==4.47.1 accelerate==1.2.1 tldextract==5.1.3
!pip -q install --no-deps -e .
import importlib
for m in ["torch", "transformers", "accelerate", "sklearn", "pandas", "pyarrow", "yaml", "tldextract"]:
    print(f"{m:13s}", importlib.import_module(m).__version__ if hasattr(importlib.import_module(m), "__version__") else "ok")


### 3. Rebuild the message split and verify it matches the committed one

In [ ]:
import json, shutil
committed = json.load(open("data/processed/messages_split_manifest.json"))
shutil.copy("data/processed/messages_split_manifest.json", "/content/committed_manifest.json")

!python -m fraudshield.data.download --only uci_sms mendeley_smishing
!python -m fraudshield.data.clean_messages
!python -m fraudshield.data.split messages

rebuilt = json.load(open("data/processed/messages_split_manifest.json"))
for name in ("train", "val", "test"):
    a, b = committed["splits"][name], rebuilt["splits"][name]
    print(f"{name:5s} rows {a['rows']} vs {b['rows']} | hash {a['hash']} vs {b['hash']}")
    assert a["hash"] == b["hash"], f"{name} split differs from the committed split: stop and investigate"
print("Splits match the committed manifest.")


### 4. Sweep (about 3 min per configuration on a T4)

In [ ]:
!python -m fraudshield.models.distilbert --sweep 2>&1 | grep -vE "it/s\]|s/it\]"


In [ ]:
import glob
res = json.load(open(sorted(glob.glob("reports/metrics/distilbert_*.json"))[-1]))
print("Best:", res["best"], "| device:", res["device"])
for k, v in sorted(res["search_val_macro_f1"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:28s} val macro-F1 {v:.4f}")
v = res["val"]
print(f"\nVal macro-F1 {v['macro_f1']:.4f} | scam recall {v['per_class']['scam']['recall']:.3f} | "
      f"ham FPR {v['fpr_benign_flagged']:.4f}")
print("Current local model (CPU run): macro-F1 0.8623 | scam recall 0.896 | ham FPR 0.0054")


### 5. Download the results

In [ ]:
metrics = sorted(glob.glob("reports/metrics/distilbert_*.json"))[-1]
!rm -f /content/distilbert_sweep_results.zip
!zip -qr /content/distilbert_sweep_results.zip models/distilbert {metrics} reports/figures/cm_distilbert_val.png -x "models/distilbert/training_args.bin"
!ls -lh /content/distilbert_sweep_results.zip
from google.colab import files
files.download("/content/distilbert_sweep_results.zip")


In [ ]:
# Optional: also keep a copy on Google Drive (e.g. to share with your teammate)
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/fraudshield
    !cp /content/distilbert_sweep_results.zip /content/drive/MyDrive/fraudshield/
